In [1]:
# =======================================================
# LightGBM Regressor를 바탕으로 생활비용지수 모델 학습
# =======================================================

In [11]:
# 1. import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib.externals.loky.backend import context

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from lightgbm import LGBMRegressor
from lightgbm import early_stopping, log_evaluation

In [3]:
# 2. 데이터 가져오기

cost_name = "생활비용_모델용_월단위_202408_202512.csv"

cost_df = pd.read_csv(cost_name, encoding="utf-8-sig")

print(f"{cost_name}의 item 갯수 {len(cost_df)}")

cost_df.head(2)


생활비용_모델용_월단위_202408_202512.csv의 item 갯수 7242


,YYYYMM,행정동코드,당월_매출_금액,당월_매출_건수,주중_매출_금액,주말_매출_금액,남성_매출_금액,여성_매출_금액,행정동이름,AREA_M2,...,아파트_면적_132_제곱미터_세대_수,아파트_면적_165_제곱미터_세대_수,아파트_평균_면적,아파트_평균_시가,M2,KOSPI,HOUSE_SALE,RENT_SALE,FX,생활인구합계
0,202408,1111051500,2.582955e+10,1218354.0,1.931113e+10,6.518425e+09,8.443738e+09,1.532099e+10,청운효자동,2438307,...,91.0,117.0,69.0,276146690,3834303.6,2647.99,91.47,96.173,1354.15,10466681
1,202408,1111053000,1.030000e+11,4456047.0,8.339905e+10,1.945317e+10,4.075866e+10,3.963525e+10,사직동,1165780,...,81.0,14.0,73.0,371237211,3834303.6,2647.99,91.47,96.173,1354.15,21399917


In [4]:
#3. 데이터 전처리
print('\n [결측치 확인]')
print(cost_df.isnull().sum())



 [결측치 확인]
YYYYMM                    0
행정동코드                     0
당월_매출_금액                  0
당월_매출_건수                  0
주중_매출_금액                  0
주말_매출_금액                  0
남성_매출_금액                  0
여성_매출_금액                  0
행정동이름                     0
AREA_M2                   0
LAT                       0
LON                       0
전체승객수                     0
지하철승객수                    0
버스승객수                     0
아파트_단지_수                  0
아파트_면적_66_제곱미터_미만_세대_수    0
아파트_면적_66_제곱미터_세대_수       0
아파트_면적_99_제곱미터_세대_수       0
아파트_면적_132_제곱미터_세대_수      0
아파트_면적_165_제곱미터_세대_수      0
아파트_평균_면적                 0
아파트_평균_시가                 0
M2                        0
KOSPI                     0
HOUSE_SALE                0
RENT_SALE                 0
FX                        0
생활인구합계                    0
dtype: int64


In [57]:
# 1차
# 생활비용지수 = 0.5×z(1인당매출)+0.3×z(아파트평균시가/가족구성명수(2))+0.15×z(전체승객수/AREA_m2)+0.05×z(아파트평균면적)

# cost_df["생활비용지수"] = 0.5*np.log1p(cost_df["당월_매출_금액"]/cost_df["생활인구합계"])     \
#                         + 0.3*np.log1p(cost_df["아파트_평균_시가"]/2)                     \
#                         + 0.15*np.log1p(cost_df["전체승객수"]/cost_df["AREA_M2"])        \
#                         + 0.05*np.log1p(cost_df["아파트_평균_면적"])
#
# cost_df.head(2)

#cost_df.to_csv("생활비용_모델용_월단위_202408_202512_costindex.csv", index=False, encoding="utf-8-sig")


In [5]:
# 2차
# 생활비용지수 (CLI) =  w1*소비강도 + w2*주거비용 + w3*수요압력 + w4*거시환경
# 소비강도 = Z((당월매출금액/생활인구합계)
# 주거비용 = Z(아파트평균시가/평균가족구성원수/아파트_평균_면적)+0.2*서울전세가격지수+0.2*서울주택매매지수
# 수요압력 = Z(전체승객수/행정동면적)
# 거시환경 = Z(M2)+Z(FX)-Z(KOSPI)
# Z는 z-score 또는 log1p 으로 튜닝
# w1=0.5, w2=0.3,w3=0.15,w4=0.05를 시작으로 튜닝
# CLI=0.5*Z((당월매출금액/생활인구합계)+0.3*(Z(아파트평균시가/평균가족구성원수/아파트_평균_면적)+0.2*서울전세가격지수+0.2*서울주택매매지수)+0.15*Z(전체승객수/행정동면적)+0.05*(Z(M2)+Z(FX)-Z(KOSPI))

cost_df["생활비용지수"] = 0.5*np.log1p(cost_df["당월_매출_금액"]/cost_df["생활인구합계"])     \
                        + 0.3*np.log1p(cost_df["아파트_평균_시가"]/2/cost_df["아파트_평균_면적"]+0.2*cost_df["HOUSE_SALE"]+0.2*cost_df["RENT_SALE"])        \
                        + 0.15*np.log1p(cost_df["전체승객수"]/cost_df["AREA_M2"])        \
                        + 0.05*(np.log1p(cost_df["M2"])+np.log1p(cost_df["FX"])-np.log1p(cost_df["KOSPI"]))

# 임시 저장용
cost_df.to_csv("생활비용_모델용_월단위_202408_202512_cli.csv", index=False, encoding="utf-8-sig")


In [9]:
# feature engineering
# 행정동에 대하여 이전 3개월 lag data 기반으로 이번달 예측
# 이전 t-3개월 lag 데이터 기반으로 이번 t달 에측

# 임시 저장에서 읽어오기
cli_df = pd.read_csv("생활비용_모델용_월단위_202408_202512_cli.csv", encoding="utf-8-sig")


cols = [
    "YYYYMM",
    "행정동코드"
    "행정동이름"
    "당월_매출_금액",
    "당월_매출_건수",
    "주중_매출_금액",
    "주말_매출_금액",
    "남성_매출_금액",
    "여성_매출_금액",
    "AREA_M2",
    "LAT",
    "LON",
    "전체승객수",
    "지하철승객수",
    "버스승객수",
    "아파트_단지_수",
    "아파트_면적_66_제곱미터_미만_세대_수",
    "아파트_면적_66_제곱미터_세대_수",
    "아파트_면적_99_제곱미터_세대_수",
    "아파트_면적_132_제곱미터_세대_수",
    "아파트_면적_165_제곱미터_세대_수",
    "아파트_평균_면적",
    "아파트_평균_시가",
    "M2",
    "KOSPI",
    "HOUSE_SALE",
    "RENT_SALE",
    "FX",
    "생활인구합계",
    "생활비용지수"
]

feat_cols = [

    'YYYYMM',

    '행정동코드',   # LightGBM categorical

    '생활비용지수',

    # ── lag 피처: t-1, t-2, t-3 (이전달까지 확정값)
    "생활비용지수_lag1",
    "생활비용지수_lag2",
    "생활비용지수_lag3",

    # ── 통계 피처 (과거 기준)
    "생활비용지수_roll3_mean",
    "생활비용지수_roll3_std",
    "생활비용지수_change1",
    "생활비용지수_change3_mean",

    # ── 시간 피처
    "YEAR",
    "MONTH",
    "MONTH_SIN",
    "MONTH_COS",

    # ── 거시 피처
    "KOSPI_lag1",
    "M2_lag1",
    "FX_lag1",
    "HOUSE_SALE_lag1",
    "RENT_SALE_lag1",

    # ── 매출·인구
    "당월_매출_금액_lag1",
    "생활인구합계_lag1",
    "전체승객수_lag1",

    # ── 부동산가격
    "아파트_평균_시가_lag1",

    # 정적 피처
    "AREA_M2",
    "LAT",
    "LON",
    "아파트_단지_수",
    "아파트_면적_66_제곱미터_미만_세대_수",
    "아파트_면적_66_제곱미터_세대_수",
    "아파트_면적_99_제곱미터_세대_수",
    "아파트_면적_132_제곱미터_세대_수",
    "아파트_면적_165_제곱미터_세대_수",
    "아파트_평균_면적",
]



# 정렬 다시 확인
cli_df = cli_df.sort_values(
    ["행정동코드", "YYYYMM"]
)

group_df = cli_df.groupby("행정동코드")

cli_df["생활비용지수_lag1"] = group_df["생활비용지수"].shift(1)
cli_df["생활비용지수_lag2"] = group_df["생활비용지수"].shift(2)
cli_df["생활비용지수_lag3"] = group_df["생활비용지수"].shift(3)

cli_df["생활비용지수_roll3_mean"] = (group_df["생활비용지수"].transform(lambda x: x.shift(1).rolling(3).mean()))
cli_df["생활비용지수_roll3_std"] = (group_df["생활비용지수"].transform(lambda x: x.shift(1).rolling(3).std()))

cli_df["생활비용지수_change1"] = group_df["생활비용지수"].shift(1).pct_change(1)
cli_df["생활비용지수_change3_mean"] = group_df["생활비용지수_change1"].transform(lambda x: x.shift(1).rolling(3).mean())

cli_df["KOSPI_lag1"] = group_df["KOSPI"].shift(1)
cli_df["M2_lag1"] = group_df["M2"].shift(1)
cli_df["FX_lag1"] = group_df["FX"].shift(1)
cli_df["HOUSE_SALE_lag1"] = group_df["HOUSE_SALE"].shift(1)
cli_df["RENT_SALE_lag1"] = group_df["RENT_SALE"].shift(1)

cli_df["당월_매출_금액_lag1"] = group_df["당월_매출_금액"].shift(1)
cli_df["생활인구합계_lag1"] = group_df["생활인구합계"].shift(1)
cli_df["전체승객수_lag1"] = group_df["전체승객수"].shift(1)
cli_df["아파트_평균_시가_lag1"] = group_df["아파트_평균_시가"].shift(1)

cli_df['YEAR']     = cli_df['YYYYMM'] // 100
cli_df['MONTH']    = cli_df['YYYYMM'] %  100
cli_df['MONTH_SIN']= np.sin(2 * np.pi * cli_df['MONTH'] / 12)
cli_df['MONTH_COS']= np.cos(2 * np.pi * cli_df['MONTH'] / 12)

cli_df = cli_df.dropna()

# 정렬 다시 확인
cli_df = cli_df.sort_values(
    [ "YYYYMM","행정동코드"]
)


feat_df = cli_df[feat_cols]

feat_df.to_csv("생활비용_학습용_202408_202512_features.csv", index=False, encoding="utf-8-sig")

print('\n [결측치 확인]')
print(feat_df.isnull().sum())

print(len(feat_df))

feat_df.head(2)



C:\Users\human\AppData\Local\Temp\ipykernel_4212\4258492699.py:111: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  cli_df["생활비용지수_change1"] = group_df["생활비용지수"].shift(1).pct_change(1)



 [결측치 확인]
YYYYMM                    0
행정동코드                     0
생활비용지수                    0
생활비용지수_lag1               0
생활비용지수_lag2               0
생활비용지수_lag3               0
생활비용지수_roll3_mean         0
생활비용지수_roll3_std          0
생활비용지수_change1            0
생활비용지수_change3_mean       0
YEAR                      0
MONTH                     0
MONTH_SIN                 0
MONTH_COS                 0
KOSPI_lag1                0
M2_lag1                   0
FX_lag1                   0
HOUSE_SALE_lag1           0
RENT_SALE_lag1            0
당월_매출_금액_lag1             0
생활인구합계_lag1               0
전체승객수_lag1                0
아파트_평균_시가_lag1            0
AREA_M2                   0
LAT                       0
LON                       0
아파트_단지_수                  0
아파트_면적_66_제곱미터_미만_세대_수    0
아파트_면적_66_제곱미터_세대_수       0
아파트_면적_99_제곱미터_세대_수       0
아파트_면적_132_제곱미터_세대_수      0
아파트_면적_165_제곱미터_세대_수      0
아파트_평균_면적                 0
dtype: int64
5962


,YYYYMM,행정동코드,생활비용지수,생활비용지수_lag1,생활비용지수_lag2,생활비용지수_lag3,생활비용지수_roll3_mean,생활비용지수_roll3_std,생활비용지수_change1,생활비용지수_change3_mean,...,AREA_M2,LAT,LON,아파트_단지_수,아파트_면적_66_제곱미터_미만_세대_수,아파트_면적_66_제곱미터_세대_수,아파트_면적_99_제곱미터_세대_수,아파트_면적_132_제곱미터_세대_수,아파트_면적_165_제곱미터_세대_수,아파트_평균_면적
1279,202411,1111053000,9.590685,9.561261,9.576046,9.555420,9.564242,0.010631,-0.001544,0.023698,...,1165780,126.97014,37.57411,89,309,158.0,54,81.0,14.0,73.0
1280,202411,1111054000,9.689516,9.633549,9.647204,9.634118,9.638291,0.007725,-0.001415,0.003086,...,1361445,126.98111,37.58801,9,12,19.0,5,2.0,14.0,111.0


In [21]:
dtype_cols = [

    '생활비용지수',

    # ── lag 피처: t-1, t-2, t-3 (이전달까지 확정값)
    "생활비용지수_lag1",
    "생활비용지수_lag2",
    "생활비용지수_lag3",

    # ── 통계 피처 (과거 기준)
    "생활비용지수_roll3_mean",
    "생활비용지수_roll3_std",
    "생활비용지수_change1",
    "생활비용지수_change3_mean",

    # ── 시간 피처
    "MONTH_SIN",
    "MONTH_COS",

    # ── 거시 피처
    "KOSPI_lag1",
    "M2_lag1",
    "FX_lag1",
    "HOUSE_SALE_lag1",
    "RENT_SALE_lag1",

    # ── 매출·인구
    "당월_매출_금액_lag1",
    "생활인구합계_lag1",
    "전체승객수_lag1",

    # ── 부동산가격
    "아파트_평균_시가_lag1",

    # 정적 피처
    "LAT",
    "LON",

    "아파트_면적_66_제곱미터_미만_세대_수",
    "아파트_면적_66_제곱미터_세대_수",
    "아파트_면적_99_제곱미터_세대_수",
    "아파트_면적_132_제곱미터_세대_수",
    "아파트_면적_165_제곱미터_세대_수",
    "아파트_평균_면적",
]


feat_df = pd.read_csv("생활비용_학습용_202408_202512_features.csv", encoding="utf-8-sig")

feat_df["행정동코드"] = feat_df["행정동코드"].astype("category")

feat_df[dtype_cols] = feat_df[dtype_cols].astype("float32")

print(feat_df[feat_cols].dtypes)



YYYYMM                       int64
행정동코드                     category
생활비용지수                     float32
생활비용지수_lag1                float32
생활비용지수_lag2                float32
생활비용지수_lag3                float32
생활비용지수_roll3_mean          float32
생활비용지수_roll3_std           float32
생활비용지수_change1             float32
생활비용지수_change3_mean        float32
YEAR                         int64
MONTH                        int64
MONTH_SIN                  float32
MONTH_COS                  float32
KOSPI_lag1                 float32
M2_lag1                    float32
FX_lag1                    float32
HOUSE_SALE_lag1            float32
RENT_SALE_lag1             float32
당월_매출_금액_lag1              float32
생활인구합계_lag1                float32
전체승객수_lag1                 float32
아파트_평균_시가_lag1             float32
AREA_M2                      int64
LAT                        float32
LON                        float32
아파트_단지_수                     int64
아파트_면적_66_제곱미터_미만_세대_수     float32
아파트_면적_66_제곱미터_세대_수 

In [22]:
# 데이터 분할
# 시계열로 분할

cost_index = "생활비용지수"

train_df = feat_df[(feat_df["YYYYMM"] >= 202411) & (feat_df["YYYYMM"] <= 202510)]
valid_df = feat_df[(feat_df["YYYYMM"] >= 202511) & (feat_df["YYYYMM"] <= 202511)]
test_df = feat_df[(feat_df["YYYYMM"] >= 202512) & (feat_df["YYYYMM"] <= 202512)]


X_train = train_df.drop(columns=cost_index)
y_train = train_df[cost_index]

X_valid = valid_df.drop(columns=cost_index)
y_valid = valid_df[cost_index]


X_test = test_df.drop(columns=cost_index)
y_test = test_df[cost_index]


#display(X_train.head(2))
print("train:", X_train.shape)

print("valid:", X_valid.shape)


#display(X_test.head(2))
print("test:", X_test.shape)

print(f"train 기간: {X_train['YYYYMM'].min()} - {X_train['YYYYMM'].max()}")
print(f"valid 기간: {X_valid['YYYYMM'].min()} - {X_valid['YYYYMM'].max()}")
print(f"test  기간: {X_test['YYYYMM'].min()} - {X_test['YYYYMM'].max()}")

train: (5110, 32)
valid: (426, 32)
test: (426, 32)
train 기간: 202411 - 202510
valid 기간: 202511 - 202511
test  기간: 202512 - 202512


In [23]:


# 4. 모델 생성 및 학습 (LightGBM 모델)
model = LGBMRegressor(
    objective="regression",
    n_estimators=3000,  # 트리개수
    learning_rate=0.03,  # 학습률
    max_depth=-1,        # 트리 최대 깊이  (-1 제한없음) 
    num_leaves=31,       # leaf 개수
    subsample=1.0,       # 데이터 샘플링 비율 (과적합 방지)
    colsample_bytree=0.9, # feature 샘플링 비율 (과적합 방지)
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="l1",
    categorical_feature=["행정동코드"],
    callbacks=[
        early_stopping(stopping_rounds=100),
        log_evaluation(period=100)
    ]
)


# 모델 학습
model.fit(X_train, y_train)    # leaf-wise 방식 학습

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000758 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5366
[LightGBM] [Info] Number of data points in the train set: 5110, number of used features: 32
[LightGBM] [Info] Start training from score 9.044983
Training until validation scores don't improve for 100 rounds
[100]	valid_0's l1: 0.0391698	valid_0's l2: 0.00279642
Early stopping, best iteration is:
[96]	valid_0's l1: 0.0390037	valid_0's l2: 0.0028853
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of cate

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.03
,n_estimators,3000
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [ ]:
# 5. 예측 수행
y_pred = model.predict(X_test)

In [ ]:
# 6. 성능 평가
# MAE
mae = mean_absolute_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R2  <- 모델 설명력 (1에 가까울수록 좋은)
r2 = r2_score(y_test, y_pred)

# 결과 출력
print('\n [LightGBM 성능]')
print(f"MAE : {mae:.4f}")
print(f"RMSE :{rmse:.4f}")
print(f"R2 : {r2:.4f}")

In [ ]:
# 7. 성능 시각화
plt.figure(figsize=(6,5))

plt.scatter(y_test, y_pred, alpha=0.4)

plt.xlabel("actual price")
plt.ylabel("predicted price")

plt.title("actual vs predicted")

plt.show()

In [ ]:
residuals = y_test - y_pred
plt.figure(figsize=(6,5))

plt.scatter(y_pred, residuals, alpha=0.4)

# 기준선 (오차 0)
plt.axhline(0)

plt.xlabel("Predicted")
plt.ylabel("residual")

plt.title("residual plot")

plt.show()


In [ ]:
plt.figure(figsize=(6,5))

plt.hist(residuals, bins=50)
plt.xlabel("Error")
plt.ylabel("Count")

plt.title("Residual Distribution")

plt.show()



In [ ]:
# 8. Feature Importance
importance = model.feature_importances_

# 중요도 정렬
indices = np.argsort(importance)

plt.figure(figsize=(6,5))

# 가로 막대 그래프
plt.barh(range(len(indices)), importance[indices])
plt.yticks(range(len(indices)), X.columns[indices])

plt.title("feature importance (LGB)")
plt.show()

In [ ]:
# 9. 모델 예측
sample_data = X_test.iloc[:5]

# 예측 수행
sample_pred = model.predict(sample_data)

print("\n [샘플 예측 결과]")
for i, pred in enumerate(sample_pred):
    print(f"{i+1}번째 샘플 예측 집값: {pred:.4f}")

print("\n[실제값]")
print(y_test.iloc[:5].values)



In [ ]:
# 10. 예측 결과 시각화
plt.figure(figsize=(8,4))
plt.plot(y_test.iloc[:20].values, label="actual", marker='o')  # 실제값
plt.plot(model.predict(X_test.iloc[:20]), label="predicted", marker='x')

plt.title("predicted vas actual")
plt.legend()
plt.show()